In [ ]:
from src.auth import AuthenticationService
from src.ndvi import extract_ndvi
from src.canopy_height import extract_canopy_height
from src.elevation import extract_elevation
from src.landcover import extract_landcover
from src.worldclim import extract_worldclim
from src.aoi import create_aoi_from_coordinates
from src.utils import *
import pandas as pd

In [ ]:
ok = AuthenticationService.authenticate(project_id="wide-office-411000")
if not ok:
    raise SystemExit("Could not initialize EE")

In [ ]:
df = pd.read_csv("input/site_data_ea (Copy).csv")

In [ ]:
fig, ax = plot_map(df, 
                   basemap='satellite', 
                   alpha=0.7, 
                   color="#1FDBDB", 
                   buffer_pct=0.1,
                   title="Site Locations")  

In [ ]:
aoi = create_aoi_from_coordinates(df, buffer_km=30)

In [ ]:
# Process NDVI
df, image_ndvi = extract_ndvi(
    df=df,
    aoi=aoi,
)

In [ ]:
df

In [ ]:
df, image_elevation = extract_elevation(df)

In [ ]:
df

In [ ]:
df, image_canopy_height = extract_canopy_height(df)

In [ ]:
df

In [ ]:
df = extract_worldclim(df)

In [ ]:
df

In [ ]:
df, image_worldcover = extract_landcover(
    df,
    buffer_meters=500,
    start_year=2020,
    end_year=2024,
    scale=20  # Lower resolution for faster processing
)

In [ ]:
image_with_indices = ee.Image.cat([
    image_ndvi,
    image_elevation,
    image_canopy_height,
    image_worldcover   
])

In [ ]:
plot_images(image_with_indices, filter_bands=['NDVI', 'elevation', 'b1', 'Map'], aoi=aoi, zoom=10)

In [ ]:
# Save result
df.to_csv("output/points.csv", index=False)
print("\nResult saved to 'points.csv'")